# 🚀 Batch Real-ESRGAN 4K Video Upscaler (Google Drive Queue)

This notebook allows you to outsource the heavy AI upscaling computation to Google Colab. 

### How it works:
1. **Mount Google Drive** to access input videos.
2. **Place your input videos** in the queue folder in your Google Drive.
3. **Run all cells**. The notebook will process every video in the input queue one-by-one, upscale it to 4K, save it to the output folder, and move the original to a "done" folder.

## 1. Mount Google Drive & Configuration

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

#@markdown ### Define Folder Locations (relative to Google Drive root)
QUEUE_INPUT_RELATIVE = "Upscale/Input" #@param {type:"string"}
QUEUE_OUTPUT_RELATIVE = "Upscale/Output" #@param {type:"string"}
QUEUE_DONE_RELATIVE = "Upscale/Done" #@param {type:"string"}

#@markdown ### Upscaling Settings
TILE_SIZE = 256 #@param [0, 128, 256, 384, 512] {type:"raw"}
CRF = 18 #@param {type:"slider", min:10, max:30, step:1}

INPUT_DIR = os.path.join("/content/drive/MyDrive", QUEUE_INPUT_RELATIVE)
OUTPUT_DIR = os.path.join("/content/drive/MyDrive", QUEUE_OUTPUT_RELATIVE)
DONE_DIR = os.path.join("/content/drive/MyDrive", QUEUE_DONE_RELATIVE)

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DONE_DIR, exist_ok=True)

print(f"📁 Input Queue Folder: {INPUT_DIR}")
print(f"📁 Output 4K Folder: {OUTPUT_DIR}")
print(f"📁 Completed Input Archive: {DONE_DIR}")

## 2. Install Dependencies
We'll install Real-ESRGAN and its dependencies. We fetch the correct `basicsr` and `realesrgan` libraries for Google Colab's GPU runtime environment.

In [ ]:
# Uninstall potentially problematic packages
!pip uninstall -y realesrgan basicsr torchaudio

# Install dependencies
!pip install -q git+https://github.com/xinntao/basicsr.git
!pip install -q realesrgan gfpgan opencv-python-headless

import torch
print(f"✓ Torch version: {torch.__version__} (CUDA: {torch.cuda.is_available()})")

## 3. Run Batch Upscale Queue

In [ ]:
import subprocess
import json
import time
import cv2
import numpy as np
import shutil
from realesrgan import RealESRGANer
from basicsr.archs.srvgg_arch import SRVGGNetCompact

# Load Model
model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_conv=16, upscale=4, act_type='prelu')
upsampler = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth",
    model=model,
    tile=TILE_SIZE,
    tile_pad=10,
    pre_pad=0,
    half=True,
)
print("✓ Real-ESRGAN anime video model loaded successfully.")

# Find videos to process
valid_extensions = (".mp4", ".mov", ".mkv", ".avi")
videos = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)])

if not videos:
    print("❌ No videos found in the input queue directory.")
else:
    print(f"Found {len(videos)} video(s) in queue:")
    for v in videos:
        print(f" - {v}")

for idx, video_name in enumerate(videos):
    input_path = os.path.join(INPUT_DIR, video_name)
    output_name = os.path.splitext(video_name)[0] + " 4k.mp4"
    output_path = os.path.join(OUTPUT_DIR, output_name)
    done_path = os.path.join(DONE_DIR, video_name)

    print(f"\n{'='*70}")
    print(f"Processing [{idx+1}/{len(videos)}]: {video_name}")
    print(f"{'='*70}")

    # Get video info
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", input_path],
        capture_output=True, text=True
    )
    info = json.loads(probe.stdout)
    video_stream = next(s for s in info["streams"] if s["codec_type"] == "video")
    fps_parts = video_stream["r_frame_rate"].split("/")
    FPS = round(int(fps_parts[0]) / int(fps_parts[1]), 3)
    src_w, src_h = int(video_stream["width"]), int(video_stream["height"])
    print(f"Source resolution: {src_w}x{src_h} @ {FPS} fps")

    total_frames_expected = None
    try:
        total_frames_expected = int(video_stream.get('nb_frames', 0))
        if total_frames_expected == 0 and 'format' in info and 'duration' in info['format']:
            duration = float(info['format']['duration'])
            total_frames_expected = int(duration * FPS)
    except Exception:
        pass

    # Local scratch paths inside Colab disk for high performance
    FRAMES_DIR = "/content/frames_input"
    FRAMES_UP_DIR = "/content/frames_upscaled"
    shutil.rmtree(FRAMES_DIR, ignore_errors=True)
    shutil.rmtree(FRAMES_UP_DIR, ignore_errors=True)
    os.makedirs(FRAMES_DIR, exist_ok=True)
    os.makedirs(FRAMES_UP_DIR, exist_ok=True)

    # Extract frames
    print("Extracting frames to scratch space...")
    start_extract = time.time()
    subprocess.run([
        "ffmpeg", "-y", "-i", input_path, 
        "-qscale:v", "2", f"{FRAMES_DIR}/frame_%06d.png", 
        "-loglevel", "warning"
    ])
    extracted_count = len([f for f in os.listdir(FRAMES_DIR) if f.endswith(".png")])
    print(f"✓ Extracted {extracted_count} frames in {time.time() - start_extract:.1f}s")

    # Upscale frames
    frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith(".png"))
    start_upscale = time.time()
    print("Starting frame upscaling...")
    
    for i, fname in enumerate(frames):
        img = cv2.imread(os.path.join(FRAMES_DIR, fname), cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        enhanced, _ = upsampler.enhance(rgb, outscale=4)
        output = cv2.cvtColor(enhanced, cv2.COLOR_RGB2BGR)

        # Resize to exactly 4K (3840x2160)
        if output.shape[1] != 3840 or output.shape[0] != 2160:
            output = cv2.resize(output, (3840, 2160), interpolation=cv2.INTER_LANCZOS4)

        cv2.imwrite(os.path.join(FRAMES_UP_DIR, fname), output)

        if (i + 1) % 50 == 0 or (i + 1) == len(frames):
            elapsed = time.time() - start_upscale
            fps_rate = (i + 1) / elapsed
            remaining_sec = (len(frames) - i - 1) / fps_rate
            print(f"  [{i+1}/{len(frames)}] {fps_rate:.2f} fps — ~{remaining_sec/60:.1f}m remaining")

    print(f"✓ Upscaled complete in {(time.time() - start_upscale)/60:.1f} minutes")

    # Reassemble frames into 4K video
    concat_path = "/content/concat_list.txt"
    frame_dur = f"{1/FPS:.10f}"
    with open(concat_path, "w") as f:
        for fname in frames:
            f.write(f"file '{FRAMES_UP_DIR}/{fname}'\n")
            f.write(f"duration {frame_dur}\n")

    print(f"Encoding to final 4K video file...")
    # Use hardware-accelerated H.264 encoder on Colab GPU
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0", "-i", concat_path,
        "-i", input_path,
        "-map", "0:v:0", "-map", "1:a:0?",
        "-c:v", "h264_nvenc", "-cq", str(CRF), "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "320k",
        "-r", str(FPS),
        "-movflags", "+faststart",
        output_path,
        "-loglevel", "warning"
    ])

    # Cleanup scratch directories for next run
    shutil.rmtree(FRAMES_DIR, ignore_errors=True)
    shutil.rmtree(FRAMES_UP_DIR, ignore_errors=True)
    if os.path.exists(concat_path):
        os.remove(concat_path)

    # Archive original input file to completed directory
    shutil.move(input_path, done_path)

    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✓ Completed! Saved 4K output to: {output_path} ({size_mb:.1f} MB)")
    print(f"✓ Archived input file to: {done_path}")

print("\n🎉 All queued videos have been successfully upscaled to 4K!")